# Custom steering vectors - BAGEL
Inspect one text pair and the reusable extraction functions. Use an A100 80 GB.


## 1. Setup
Publish this branch before opening Colab. Run setup in a fresh runtime.


In [ ]:
from pathlib import Path

PROJECT = Path("/content/bagel-space")
UPSTREAM = Path("/content/BAGEL")
if not PROJECT.exists():
    !git clone --branch experiment/custom-steering-vectors https://github.com/Nicerova7/bagel-space.git /content/bagel-space
if not UPSTREAM.exists():
    !git clone https://github.com/ByteDance-Seed/Bagel.git /content/BAGEL
!git -C /content/bagel-space fetch origin
!git -C /content/bagel-space checkout experiment/custom-steering-vectors
!git -C /content/BAGEL checkout a2fa77dd8caeefc41e6607ae0ec17408d3f4ee9f
%pip install -r /content/bagel-space/requirements.txt


### Install FlashAttention
Install the Astral binary for Python 3.13, PyTorch 2.11, and CUDA 13.0.


In [ ]:
%pip install --no-deps "https://github.com/astral-sh-build/build-flash-attention/releases/download/v2.8.3-r4/flash_attn-2.8.3%2Bcu.13.0.torch.2.11-cp313-cp313-manylinux_2_24_x86_64.whl"
print("Restart the session, then continue below.")


## 2. Load BAGEL
Restart after installation, then run once.


In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

import torch
from IPython.display import display

PROJECT = Path("/content/bagel-space")
UPSTREAM = Path("/content/BAGEL")
os.chdir(PROJECT)
os.environ["PYTHONPATH"] = str(UPSTREAM) + os.pathsep + str(PROJECT)
sys.path[:0] = [str(PROJECT), str(UPSTREAM)]

from bagel_space.adapter import BagelAdapter
from bagel_space.capture import Capture

adapter = BagelAdapter()
print(torch.cuda.get_device_name(0), adapter.dtype)
print("Layers:", len(adapter.layers), "| Hidden size:", adapter.model.hidden_size)
from experiments.custom_utils import (
    prepare_pair, capture_pair, build_steering_vector, make_steering_vector,
)


## 3. Define and inspect one text pair
Shared BOS, no EOS; `right_pad` appends visible spaces to equalize token counts.


In [ ]:
pair_pos = "A red car parked on a street."
pair_neg = "A blue car parked on a street."
alignment = "right_pad"   # "none", "right_pad"

pair = prepare_pair(adapter, pair_pos, pair_neg, alignment=alignment)
for side, ids in pair["token_ids"].items():
    print(side, adapter.tokenizer.convert_ids_to_tokens(ids))
print("Padding:", pair["padding"])


## 4. Capture selected layers
Use an index, list, slice, or `all`. Each layer remains separate.


In [ ]:
layers = "all"
captured = capture_pair(adapter, pair, layers=layers)
for layer in captured["layers"]:
    print("Layer:", layer, "| Positive:", captured["positive"][layer].shape,
          "| Negative:", captured["negative"][layer].shape)


## 5. Select tokens and construct the direction
Positive minus negative; selections include BOS. With `last`, mean pooling keeps that single state.


In [ ]:
tokens = "last"           # "first", "last", "all", index, list, slice
token_reduction = "mean"  # "none", "sum", "mean", callable
layer_reduction = "none"
direction_method = "difference"

result = build_steering_vector(
    captured, tokens=tokens, token_reduction=token_reduction,
    layer_reduction=layer_reduction, direction_method=direction_method,
)
layer = captured["layers"][0]
print("Positive:", result["positive"][layer].reshape(-1)[:6])
print("Negative:", result["negative"][layer].reshape(-1)[:6])
print("Difference:", result["vectors"][layer].reshape(-1)[:6])
print("Shape:", result["vectors"][layer].shape)


## 6. Compare token reductions
Reuse the captured pair; these average tokens, not multiple examples.


In [ ]:
mean_result = build_steering_vector(captured, tokens="all", token_reduction="mean")
sum_result = build_steering_vector(captured, tokens="all", token_reduction="sum")

print("Token counts:", len(pair["token_ids"]["positive"]), len(pair["token_ids"]["negative"]))
print("Mean difference:", mean_result["vectors"][layer][:6])
print("Sum difference:", sum_result["vectors"][layer][:6])


## 7. Try a custom reducer
A reducer receives `states` and `dim`; this example averages the last two selected tokens.


In [ ]:
def last_two_mean(states, *, dim):
    return states[-2:].mean(dim=dim)

custom_result = build_steering_vector(
    captured, tokens="all", token_reduction=last_two_mean,
)
print("Custom difference:", custom_result["vectors"][layer][:6])


## 8. Use the single-call helper
Runs two new forwards; compare with the result from step 5.


In [ ]:
single_call = make_steering_vector(
    adapter, pair_pos=pair_pos, pair_neg=pair_neg, alignment=alignment,
    tokens=tokens, layers=layers, token_reduction=token_reduction,
    layer_reduction=layer_reduction, direction_method=direction_method,
)
print("Same direction:", torch.allclose(single_call["vectors"][layer], result["vectors"][layer]))


## 9. Save and download
Save the step 5 result and full captures. Dataset building and application are in the counting notebook.


In [ ]:
from google.colab import files

OUTPUT = PROJECT / "outputs/custom-steering"
OUTPUT.mkdir(parents=True, exist_ok=True)
metadata = dict(result["metadata"], model=adapter.metadata(), tokens=repr(tokens))
torch.save(dict(result=result, captured=captured, metadata=metadata), OUTPUT / "steering_vector.pt")
(OUTPUT / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
archive = shutil.make_archive("/content/bagel-custom-steering", "zip", OUTPUT)
files.download(archive)
